# BIOSTAT 707 Checkpoint 1

**Rosella Zheng**  
**NetID: yz1026**


## Setup and data loading


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
DATA = ROOT / "data"
SET_A = DATA / "set-a"
OUT = ROOT / "output"

OUT.mkdir(exist_ok=True)

files = sorted(SET_A.glob("*.txt"))

print("Number of set-a files:", len(files))
print("Outcomes file exists:", (DATA / "Outcomes-a.txt").exists())

assert len(files) == 4000
assert (DATA / "Outcomes-a.txt").exists()


## Construct the long-format dataset


In [ ]:
long_list = []

for file in files:
    df = pd.read_csv(file)
    record_id = int(df.loc[df["Parameter"] == "RecordID", "Value"].iloc[0])
    df["RecordID"] = record_id
    long_list.append(df)

set_a_long = pd.concat(long_list, ignore_index=True)

print("Long table shape:", set_a_long.shape)
set_a_long.head()


In [ ]:
print("Columns:", list(set_a_long.columns))
print("Number of -1 values:", (set_a_long["Value"] == -1).sum())


In [ ]:
set_a_long_clean = set_a_long.copy()
set_a_long_clean.loc[
    set_a_long_clean["Value"] == -1,
    "Value"
] = np.nan

def time_to_hours(t):
    hours, minutes = t.split(":")
    return int(hours) + int(minutes) / 60

set_a_long_clean["Hour"] = set_a_long_clean["Time"].apply(time_to_hours)

set_a_long_clean["Hour"].describe()


The files contain irregularly timed measurements within the observation window. I converted the dataset's missing-value code to `NaN` before summarizing measurements.


## Data quality checks


In [ ]:
range_summary = (
    set_a_long_clean
    .groupby("Parameter")["Value"]
    .agg(["count", "min", "median", "max"])
    .sort_index()
)

range_summary


In [ ]:
important_vars = [
    "Age", "Gender", "Height", "ICUType", "Weight",
    "GCS", "FiO2", "pH", "Temp",
    "HR", "RespRate",
    "SysABP", "DiasABP", "MAP",
    "NISysABP", "NIDiasABP", "NIMAP",
    "SaO2"
]

range_summary.loc[
    range_summary.index.intersection(important_vars)
]


In [ ]:
suspect = set_a_long_clean[
    ((set_a_long_clean["Parameter"] == "pH") &
     ((set_a_long_clean["Value"] < 5) | (set_a_long_clean["Value"] > 10))) |
    ((set_a_long_clean["Parameter"] == "Temp") &
     ((set_a_long_clean["Value"] < 20) | (set_a_long_clean["Value"] > 45))) |
    ((set_a_long_clean["Parameter"] == "Height") &
     ((set_a_long_clean["Value"] < 100) | (set_a_long_clean["Value"] > 250))) |
    ((set_a_long_clean["Parameter"] == "Weight") &
     (set_a_long_clean["Value"] <= 0))
].copy()

print("Clearly implausible values flagged:", len(suspect))
suspect.sort_values(["Parameter", "Value"]).head(30)


I inspected variable ranges before making any additional cleaning decisions. I treated only values that were clearly incompatible with the documented units as missing, including extreme pH values, implausible temperatures and heights, and nonpositive body weights. I kept other extreme ICU measurements because they could reflect severe illness or measurement artifacts that are not obviously distinguishable from real values.


In [ ]:
analysis_long = set_a_long_clean.copy()

analysis_long.loc[
    (analysis_long["Parameter"] == "pH") &
    ((analysis_long["Value"] < 5) | (analysis_long["Value"] > 10)),
    "Value"
] = np.nan

analysis_long.loc[
    (analysis_long["Parameter"] == "Temp") &
    ((analysis_long["Value"] < 20) | (analysis_long["Value"] > 45)),
    "Value"
] = np.nan

analysis_long.loc[
    (analysis_long["Parameter"] == "Height") &
    ((analysis_long["Value"] < 100) | (analysis_long["Value"] > 250)),
    "Value"
] = np.nan

analysis_long.loc[
    (analysis_long["Parameter"] == "Weight") &
    (analysis_long["Value"] <= 0),
    "Value"
] = np.nan

analysis_long.to_csv(OUT / "set-a_long.csv", index=False)

print("Saved:", OUT / "set-a_long.csv")


## Temporal missingness


In [ ]:
temporal_data = analysis_long[
    analysis_long["Value"].notna()
].copy()

temporal_data["hour_bin"] = pd.cut(
    temporal_data["Hour"],
    bins=[0, 6, 12, 18, 24, 30, 36, 42, 48],
    include_lowest=True
)

temporal_summary = (
    temporal_data
    .groupby(["Parameter", "hour_bin"], observed=True)["RecordID"]
    .nunique()
    .reset_index(name="n_measured")
)

temporal_summary["proportion_measured"] = (
    temporal_summary["n_measured"] / len(files)
)

temporal_heatmap = temporal_summary.pivot(
    index="Parameter",
    columns="hour_bin",
    values="proportion_measured"
).fillna(0)

static_vars = ["RecordID", "Age", "Gender", "Height", "ICUType"]

temporal_heatmap_plot = temporal_heatmap.drop(
    index=static_vars,
    errors="ignore"
)

temporal_summary.sort_values(
    "proportion_measured",
    ascending=False
).head(20)


In [ ]:
plt.figure(figsize=(12, 11))

sns.heatmap(
    temporal_heatmap_plot,
    cmap="viridis",
    vmin=0,
    vmax=1
)

plt.title("Proportion of ICU Stays with Variable Measured Over Time")
plt.xlabel("Hours Since ICU Admission")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()


Measurement frequency differs substantially across variables and across time. Common bedside measurements are recorded much more often than many laboratory tests, so the missing data are not distributed uniformly.


For this checkpoint, I summarize measurements over the full 48-hour observation window because the Challenge predictors are based on the first 48 hours of the ICU stay. I use shorter time bins in the temporal missingness plot to examine how availability changes within that window. If a future model were intended to make predictions after only the first 24 hours, measurements from the second 24 hours would need to be excluded to avoid using information that would not yet be available.

## Construct the wide-format dataset


In [ ]:
time_series = analysis_long[
    analysis_long["Parameter"] != "RecordID"
].copy()

wide_summary = (
    time_series
    .sort_values(["RecordID", "Parameter", "Hour"])
    .groupby(["RecordID", "Parameter"])["Value"]
    .agg(
        count="count",
        first="first",
        last="last",
        min="min",
        max="max",
        mean="mean"
    )
)

wide_summary.head()


In [ ]:
set_a_wide = wide_summary.unstack("Parameter")

set_a_wide.columns = [
    f"{parameter}_{stat}"
    for stat, parameter in set_a_wide.columns
]

set_a_wide = set_a_wide.reset_index()

count_cols = [
    col for col in set_a_wide.columns
    if col.endswith("_count")
]

set_a_wide[count_cols] = set_a_wide[count_cols].fillna(0)

print("Wide summary shape before static variables:", set_a_wide.shape)
set_a_wide.head()


In [ ]:
static = (
    analysis_long[
        analysis_long["Parameter"].isin(
            ["Age", "Gender", "Height", "ICUType"]
        )
    ]
    .pivot_table(
        index="RecordID",
        columns="Parameter",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

set_a_wide = static.merge(
    set_a_wide,
    on="RecordID",
    how="left",
    validate="one_to_one"
)

print("Wide table shape before outcomes:", set_a_wide.shape)
print("Unique RecordIDs:", set_a_wide["RecordID"].nunique())


The wide table has one row per ICU stay. Static admission variables are kept directly, while repeated measurements are summarized using count, first, last, minimum, maximum, and mean.


## Merge outcomes


In [ ]:
outcomes = pd.read_csv(DATA / "Outcomes-a.txt")

print("Outcome columns:", list(outcomes.columns))
outcomes.head()


In [ ]:
set_a_wide = set_a_wide.merge(
    outcomes,
    on="RecordID",
    how="left",
    validate="one_to_one"
)

outcome_vars = [
    "SAPS-I",
    "SOFA",
    "Length_of_stay",
    "Survival"
]

set_a_wide[outcome_vars] = (
    set_a_wide[outcome_vars]
    .replace(-1, np.nan)
)

print("Final wide table shape:", set_a_wide.shape)
print("Unique RecordIDs:", set_a_wide["RecordID"].nunique())
print(set_a_wide["In-hospital_death"].value_counts(dropna=False))

set_a_wide.to_csv(
    OUT / "set-a_wide.csv",
    index=False
)


The physiologic measurements are the information available during the observation window. The outcome file is merged separately and is used here for cohort description and outcome comparisons rather than as a predictor.


## Cohort characteristics


In [ ]:
def median_iqr(x):
    x = x.dropna()
    return f"{x.median():.1f} [{x.quantile(0.25):.1f}, {x.quantile(0.75):.1f}]"

def n_percent(series, value):
    valid = series.dropna()
    n = (valid == value).sum()
    pct = 100 * n / len(valid)
    return f"{n} ({pct:.1f}%)"

survived = set_a_wide["In-hospital_death"] == 0
died = set_a_wide["In-hospital_death"] == 1

rows = []

continuous_vars = {
    "Age_first": "Age",
    "Height_first": "Height",
    "Weight_first": "Weight",
    "SAPS-I": "SAPS-I",
    "SOFA": "SOFA"
}

for col, label in continuous_vars.items():
    rows.append({
        "Variable": label,
        "Overall": median_iqr(set_a_wide[col]),
        "Survived": median_iqr(set_a_wide.loc[survived, col]),
        "Died": median_iqr(set_a_wide.loc[died, col])
    })

for value, label in [(1, "Male"), (0, "Female")]:
    rows.append({
        "Variable": label,
        "Overall": n_percent(set_a_wide["Gender_first"], value),
        "Survived": n_percent(set_a_wide.loc[survived, "Gender_first"], value),
        "Died": n_percent(set_a_wide.loc[died, "Gender_first"], value)
    })

for icu_type in [1, 2, 3, 4]:
    rows.append({
        "Variable": f"ICUType {icu_type}",
        "Overall": n_percent(set_a_wide["ICUType_first"], icu_type),
        "Survived": n_percent(set_a_wide.loc[survived, "ICUType_first"], icu_type),
        "Died": n_percent(set_a_wide.loc[died, "ICUType_first"], icu_type)
    })

table1 = pd.DataFrame(rows)

table1

Continuous variables are summarized with the median and interquartile range, while categorical variables are summarized with counts and percentages.


## Outcome summary


In [ ]:
death_counts = (
    set_a_wide["In-hospital_death"]
    .value_counts()
    .sort_index()
)

death_percent = (
    set_a_wide["In-hospital_death"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

outcome_summary = pd.DataFrame({
    "n": death_counts,
    "percent": death_percent
})

outcome_summary.index = ["Survived", "Died"]

outcome_summary


In [ ]:
set_a_wide[
    ["SAPS-I", "SOFA", "Length_of_stay", "Survival"]
].describe()


In [ ]:
outcome_summary["n"].plot(
    kind="bar",
    figsize=(6, 4)
)

plt.title("In-Hospital Outcome")
plt.xlabel("")
plt.ylabel("Number of ICU Stays")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Missingness in the wide dataset


In [ ]:
count_cols = [
    col for col in set_a_wide.columns
    if col.endswith("_count")
]

missing_by_variable = pd.Series({
    col.replace("_count", ""):
    (set_a_wide[col] == 0).mean()
    for col in count_cols
}).sort_values(ascending=False)

missing_by_variable.head(20)


In [ ]:
plt.figure(figsize=(10, 8))

missing_by_variable.head(20).sort_values().plot(
    kind="barh"
)

plt.xlabel("Proportion Never Measured")
plt.ylabel("Variable")
plt.title("Variables Most Often Never Measured")
plt.tight_layout()
plt.show()


In [ ]:
ever_measured = (
    set_a_wide
    .set_index("RecordID")[count_cols]
    .gt(0)
)

ever_measured.columns = [
    col.replace("_count", "")
    for col in ever_measured.columns
]

top_vars = missing_by_variable.head(20).index
missing_map = ~ever_measured[top_vars]

plt.figure(figsize=(12, 8))

sns.heatmap(
    missing_map.sample(
        n=min(500, len(missing_map)),
        random_state=707
    ),
    cbar=False,
    yticklabels=False
)

plt.xlabel("Variable")
plt.ylabel("Sample of ICU Stays")
plt.title("Missingness Pattern in the Wide Dataset")
plt.tight_layout()
plt.show()


The wide-format view shows that some variables are almost always measured while others are absent for many ICU stays. Looking at whether a variable was ever measured avoids counting the same missing variable repeatedly across its summary columns.


## Is missingness informative?


In [ ]:
measurement_vars = [
    "Lactate",
    "TroponinT",
    "Bilirubin",
    "Albumin",
    "PaO2"
]

rows = []

for var in measurement_vars:
    measured_ids = set(
        analysis_long.loc[
            (analysis_long["Parameter"] == var) &
            (analysis_long["Value"].notna()),
            "RecordID"
        ]
    )

    temp = set_a_wide[
        ["RecordID", "In-hospital_death"]
    ].copy()

    temp["Measured"] = temp["RecordID"].isin(measured_ids)

    summary = (
        temp.groupby("Measured")["In-hospital_death"]
        .agg(["count", "mean"])
        .reset_index()
    )

    summary["Variable"] = var
    rows.append(summary)

missingness_outcome = pd.concat(
    rows,
    ignore_index=True
)

missingness_outcome["Mortality (%)"] = (
    100 * missingness_outcome["mean"]
)

missingness_outcome[
    ["Variable", "Measured", "count", "Mortality (%)"]
]


In [ ]:
plt.figure(figsize=(9, 6))

sns.barplot(
    data=missingness_outcome,
    x="Variable",
    y="Mortality (%)",
    hue="Measured"
)

plt.ylabel("In-Hospital Mortality (%)")
plt.xlabel("")
plt.title("Mortality by Whether a Variable Was Measured")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


Mortality differs between patients with and without measurements for some variables. This suggests that missingness may contain clinical information, for example because some tests are ordered more often in sicker patients. These comparisons are exploratory and do not establish why a measurement was missing.


## Challenge assessment criteria


The mortality prediction task is evaluated using both discrimination and calibration. One criterion uses the smaller of sensitivity and positive predictive value, so poor performance on either component lowers the score. The other criterion is based on a modified Hosmer-Lemeshow statistic comparing predicted and observed mortality across risk groups. No prediction model is fit at this checkpoint, so these criteria are not calculated here.


## Use of AI


Generative AI tools (ChatGPT, GitHub Codex) were used solely for code assistance, including debugging error traces, notebook structuring, and syntax reference. All suggested logic—particularly regarding missing-data summaries and longitudinal measures—was manually evaluated and verified against the dataset. Discrepancies and inefficient suggestions were corrected independently, and all submitted code reflects my own review and validation.